To generate tasks for padding I use two sources:
- current validation subset
- generate examples from claude, gpt-4 and gpt-5 with different behavior and category

In [1]:
from mountain.agentharm.utils import load_dataset
import pandas as pd

/Users/tsimur.hadeliya/code/algoverse_mountain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
val_harm_dataset = load_dataset("harmful", split="val")
val_benign_dataset = load_dataset("benign", split="val")

/Users/tsimur.hadeliya/code/algoverse_mountain/.venv/lib/python3.13/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 1950.69it/s]


In [3]:
def transform_dataset(dataset, task_type) -> list[dict]:
    out = []
    for sample in dataset.samples:
        d = {
            "task_description": sample.input,
            "task_type": task_type,
            "detailed_prompt": sample.metadata["detailed_prompt"],
            "hint_included": sample.metadata["hint_included"],
            "category": sample.metadata["category"],
            "name": sample.metadata["name"],
            "split": "val"
        }
        out.append(d)
    return out

In [4]:
val_harm_list = transform_dataset(val_harm_dataset, "harmful")
val_benign_list = transform_dataset(val_benign_dataset, "benign")

In [5]:
df = pd.DataFrame(val_harm_list + val_benign_list)

In [7]:
df.to_json("multi_task.json")

In [8]:
from tiktoken import encoding_for_model, get_encoding

In [13]:
encode = encoding_for_model("gpt-4o-mini")

In [14]:
encode.encode("Asdasdasd")

[2305, 28922, 28922, 67]

In [24]:
sample2toklen = {}
for mt_len in [5, 10, 30, 50]:
    sample2toklen[mt_len] = int(df["task_description"].apply(lambda x: len(encode.encode(x))).sample(mt_len, random_state=3).sum())

In [25]:
sample2toklen

{5: 282, 10: 464, 30: 1617, 50: 2641}